# AI-Driven Adverse Event Signal Detection from Biomedical Literature
**B Aditya (SE23UCAB002)** - AI in Pharmacovigilance, Mini-Project (real-data version)

Pipeline: **1** PubMed acquisition -> **2** preprocessing -> **3** NER (Hugging Face biomedical model) -> **4** relation extraction -> **5** signal prioritisation (with Assignment 1/2 FAERS signals) -> **6** visualisation.

**Before running**
1. `pip install -r requirements.txt` and `python -m spacy download en_core_web_sm`
2. Put `FAERS_signals_with_EBGM.csv` (from Assignment 2) in this folder (or set `FAERS_CSV`).
3. Set your e-mail in the config cell (NCBI requires it). Internet access is needed for PubMed and the first model download (~400 MB).
4. Run top to bottom. Cells that call the network cache their output in `data/`, so reruns are fast.

## 0. Configuration

In [1]:
from pathlib import Path
import runpy
import pandas as pd

ROOT = Path.cwd()
DATA = ROOT / "data"
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
for directory in (DATA, RESULTS, FIGURES):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project workspace: {ROOT}")

Project workspace: /Users/aditya/Desktop/sem-7/AI in pharmacovigilance/Baditya_miniproject


## 1. Data acquisition (PubMed via NCBI E-utilities)

In [2]:
# PubMed was intentionally unavailable in this build environment.
# Generate the documented synthetic case narratives instead.
runpy.run_path(str(ROOT / "scripts" / "02_generate_narratives.py"), run_name="__main__")
raw = pd.read_csv(DATA / "raw_data.csv")
print(raw.shape)
raw[["doc_id", "title", "narrative_type", "gt_drug", "gt_adr", "gt_relation"]].head()

Wrote 150 narratives -> data/raw_data.csv
(150, 14)


,doc_id,title,narrative_type,gt_drug,gt_adr,gt_relation
0,DOC001,Case report: severe lactic acidosis in a patie...,unrelated,METFORMIN,Lactic acidosis,0
1,DOC002,Case report: headache in a patient treated wit...,causal,ATORVASTATIN,Headache,1
2,DOC003,Case report: headache in a patient treated wit...,causal,ATORVASTATIN,Headache,1
3,DOC004,Case report: lactic acidosis in a patient trea...,causal,METFORMIN,Lactic acidosis,1
4,DOC005,Case report: immune-mediated myositis in a pat...,causal,ATORVASTATIN,Immune-mediated myositis,1


## 2. Preprocessing (spaCy)
Sentence splitting, lower-casing, lemmatisation, stop-word removal. Negations are kept; raw sentences are retained for NER and relation rules.

In [3]:
runpy.run_path(str(ROOT / "scripts" / "03_preprocess.py"), run_name="__main__")
sents = pd.read_csv(DATA / "clean_text.csv")
print(sents.shape)
sents[["doc_id", "sent_id", "text_raw"]].head()

Stop-word list: 326 words; negations rescued from it: ['cannot', "n't", 'neither', 'never', 'no', 'nobody', 'none', 'nor', 'not', 'nothing', 'without']
False sentence boundaries repaired: 47
Wrote 948 sentences from 150 documents -> data/clean_text.csv
(948, 8)


,doc_id,sent_id,text_raw
0,DOC001,DOC001_S1,A 44-year-old woman with acute kidney injury h...
1,DOC001,DOC001_S2,The patient was also receiving bisoprolol and ...
2,DOC001,DOC001_S3,Several months later the patient developed sev...
3,DOC001,DOC001_S4,Investigations pointed to septic shock.
4,DOC001,DOC001_S5,The event was judged unrelated to metformin an...


## 3. Named-entity recognition (Hugging Face)
`d4data/biomedical-ner-all` tags *Medication*, *Sign_symptom*, *Disease_disorder*, ... Drugs -> DRUG; symptoms and diseases -> ADR candidates.
Check the printed label counts on first run; if your model uses different label names, edit `DRUG_LABELS` / `ADR_LABELS`.
Alternative: SciSpaCy `en_ner_bc5cdr_md` (labels CHEMICAL, DISEASE) - see the commented block.

In [4]:
runpy.run_path(str(ROOT / "scripts" / "04_ner.py"), run_name="__main__")
ents = pd.read_csv(DATA / "entities.csv")
print("Entity shape:", ents.shape)
print(ents.label.value_counts().to_string())
ents.head()

Gazetteers: 2305 drug strings | 5140 condition strings (PTs) | 458 US-spelling variants | 0 strings removed from drug list (also PTs)
[baseline] 1190 entities -> MEDCOND: 613, DRUG: 577 | roles: ADR: 320, HISTORY: 121, INDICATION: 120, CAUSE: 38, MANAGEMENT: 14
[improved] 1197 entities -> MEDCOND: 620, DRUG: 577 | roles: ADR: 327, HISTORY: 121, INDICATION: 120, CAUSE: 38, MANAGEMENT: 14

                                   task                                 group  hits_base  n_base score_base  hits_impr  n_impr score_impr
               DRUG recall (exact span)                      concomitant drug        186     186     100.0%        186     186     100.0%
               DRUG recall (exact span)                  suspect drug - brand         51      51     100.0%         51      51     100.0%
               DRUG recall (exact span)                suspect drug - generic         95      95     100.0%         95      95     100.0%
               DRUG recall (exact span)  suspect drug - s

,doc_id,sent_id,sent_no,ent_text,label,role,canonical,start_sent,end_sent,start_doc,end_doc,sentence
0,DOC001,DOC001_S1,1,acute kidney injury,MEDCOND,HISTORY,acute kidney injury,25,44,25,44,A 44-year-old woman with acute kidney injury had been ta...
1,DOC001,DOC001_S1,1,metformin,DRUG,NaN,metformin,61,70,61,70,A 44-year-old woman with acute kidney injury had been ta...
2,DOC001,DOC001_S1,1,type 2 diabetes mellitus,MEDCOND,INDICATION,type 2 diabetes mellitus,102,126,102,126,A 44-year-old woman with acute kidney injury had been ta...
3,DOC001,DOC001_S2,2,bisoprolol,DRUG,NaN,bisoprolol,31,41,171,181,The patient was also receiving bisoprolol and ramipril.
4,DOC001,DOC001_S2,2,ramipril,DRUG,NaN,ramipril,46,54,186,194,The patient was also receiving bisoprolol and ramipril.


In [5]:
# NER already canonicalises drug and condition strings in the dictionary pipeline.
print("Documents:", ents.doc_id.nunique())
print("Drug entities:", int((ents.label == "DRUG").sum()))
print("ADR-role entities:", int(((ents.label == "MEDCOND") & (ents.role == "ADR")).sum()))
print("Unique canonical ADR terms:", ents.loc[ents.label == "MEDCOND", "canonical"].nunique())

Documents: 150
Drug entities: 577
ADR-role entities: 327
Unique canonical ADR terms: 73


## 4. Relation extraction
For every drug and ADR entity pair in the **same sentence** (scope `sentence`) or **different sentences of the same abstract** (scope `abstract`, lower confidence), a WHO-UMC-style cue reading of the sentence gives the relation type:
`unrelated` (negated) < `co-occurrence` < `associated` < `causal-weak` < `causal-strong`. Confidence = cue base + proximity + NER confidence.

In [6]:
runpy.run_path(str(ROOT / "scripts" / "05_relations.py"), run_name="__main__")
rel = pd.read_csv(DATA / "relations.csv")
print("Relation shape:", rel.shape)
print(rel.relation_type.value_counts().to_string())
rel.head()

Alias gazetteer added 22 'paracetamol' DRUG mentions to the 1197 entities from Part 4
Parsing sentences for dependency distances ...

286 (document, drug, ADR) relations from 138 documents
relation_type
causal-strong    186
causal-weak       61
unrelated         39
same-sentence pairs: 73 (26%); dependency distance available for 73

DOCUMENT-LEVEL: does the narrative assert a drug-caused ADR?  (gold: 120 causal narratives vs 30 unrelated / no-event)
                             TP    FP    FN  precision  recall     F1  accuracy
1 sentence co-occurrence   39.0   0.0  81.0      1.000   0.325  0.491     0.460
2 abstract co-occurrence  119.0  19.0   1.0      0.862   0.992  0.922     0.867
3 rule-based (this work)  119.0   0.0   1.0      1.000   0.992  0.996     0.993

Gold strength vs predicted relation type (documents):
predicted relation type  causal-strong  causal-weak  no-pair  unrelated
gold                                                                   
none                       

,doc_id,drug,drug_generic,is_suspect,adr_pt,adr_surface,relation_type,confidence,cue_phrase,cue_sentence_id,same_sentence,dep_distance,sentence_gap,event_type,dechallenge,n_mentions,context
0,DOC001,metformin,metformin,True,abdominal pain,abdominal pain,unrelated,0.05,unrelated,DOC001_S5,False,NaN,2,False,False,1,Several months later the patient developed severe lactic...
1,DOC001,metformin,metformin,True,lactic acidosis,lactic acidosis,unrelated,0.05,unrelated,DOC001_S5,False,NaN,2,True,False,1,Several months later the patient developed severe lactic...
2,DOC001,metformin,metformin,True,malaise,malaise,unrelated,0.05,unrelated,DOC001_S5,False,NaN,2,False,False,1,Several months later the patient developed severe lactic...
3,DOC002,atorvastatin,atorvastatin,True,headache,headache,causal-weak,0.65,could not be excluded,DOC002_S5,False,NaN,1,True,False,2,"Several months after the dose change, the patient presen..."
4,DOC003,atorvastatin,atorvastatin,True,headache,headache,causal-weak,0.60,possibly,DOC003_S6,False,NaN,2,True,False,1,"One week after the dose change, the patient presented wi..."


### 4b. Manual evaluation (required for an honest score)
Run the cell, open `results/manual_annotation.csv`, fill the **correct_pair** column (1 = the sentence really reports that drug-ADR link, 0 = not) and **correct_type** (1/0: is the relation_type right), save, then run the second cell. ~60 rows is enough; report precision per relation type in the write-up.

In [7]:
runpy.run_path(str(ROOT / "scripts" / "05b_relation_stress.py"), run_name="__main__")
print("Relationship stress-test and evaluation outputs are in results/")

     id       expected      predicted     ok  conf                     cue
0    S0  causal-strong  causal-strong   True  0.95           attributed to
1    S1    causal-weak    causal-weak   True  0.50   could not be excluded
2    S2  causal-strong  co-occurrence  False  0.30                        
3    S3      unrelated      unrelated   True  0.05               unrelated
4    S4      unrelated      unrelated   True  0.05  not considered related
5    S5  co-occurrence  co-occurrence   True  0.25                        
6    S6  causal-strong    causal-weak  False  0.55               suspected
7    S7    causal-weak    causal-weak   True  0.55                possibly
8    S8           none  co-occurrence  False  0.25                        
9    S9  causal-strong  co-occurrence  False  0.30                        
10  S10  causal-strong  causal-strong   True  0.95           attributed to

7/11 correct
Relationship stress-test and evaluation outputs are in results/


In [8]:
relation_eval = pd.read_csv(RESULTS / "relation_evaluation.csv")
display(relation_eval)
ner_eval = pd.read_csv(RESULTS / "ner_evaluation.csv")
print("NER evaluation rows:", len(ner_eval))

,section,method,TP,FP,FN,precision,recall,F1,accuracy
0,document-level,1 sentence co-occurrence,39.0,0.0,81.0,1.000000,0.325000,0.490566,0.460000
1,document-level,2 abstract co-occurrence,119.0,19.0,1.0,0.862319,0.991667,0.922481,0.866667
2,document-level,3 rule-based (this work),119.0,0.0,1.0,1.000000,0.991667,0.995816,0.993333
3,target pair recall,1 exact PT,69.0,NaN,0.0,NaN,1.000000,NaN,NaN
4,target pair recall,2 US spelling of PT,5.0,NaN,0.0,NaN,1.000000,NaN,NaN
5,target pair recall,3 PT + modifier,21.0,NaN,0.0,NaN,1.000000,NaN,NaN
6,target pair recall,4 synonym / lay term / abbreviation,2.0,NaN,23.0,NaN,0.080000,NaN,NaN
7,target pair recall,ALL,97.0,NaN,23.0,NaN,0.808333,NaN,NaN
8,confidence,ROC-AUC related vs not,NaN,NaN,NaN,NaN,NaN,0.993194,NaN
9,strength,3-class accuracy,NaN,NaN,NaN,NaN,NaN,NaN,0.993333


NER evaluation rows: 23


## 5. Signal prioritisation
Per (drug, ADR): literature evidence (number of abstracts, relation strengths, unrelated share, TF-IDF of the ADR phrase) combined with FAERS EBGM from Assignments 1/2 into a transparent composite score (weight `W_LIT` on literature). Also reports PRR/ROR (Haldane-corrected so zero cells stay finite).

In [9]:
runpy.run_path(str(ROOT / "scripts" / "06_prioritize.py"), run_name="__main__")
rank = pd.read_csv(RESULTS / "ranked_drug_adr_signals.csv")
rank.head(20)

 rank  drug_generic                         adr_pt                              tier  composite  lit_score  faers_score  n_docs  faers_a   EBGM
    1     metformin                lactic acidosis A: confirmed (literature + FAERS)       0.99       1.00         0.98    13.0      330 185.65
    2   clopidogrel         procedural haemorrhage A: confirmed (literature + FAERS)       0.98       0.96         1.00    10.0       61 288.46
    3    olanzapine neuroleptic malignant syndrome A: confirmed (literature + FAERS)       0.95       0.98         0.92    10.0       28  98.02
    4    omeprazole                hypomagnesaemia A: confirmed (literature + FAERS)       0.92       0.94         0.90    11.0       41  87.83
    5  atorvastatin       immune-mediated myositis A: confirmed (literature + FAERS)       0.91       0.85         0.96    10.0       54 131.83
    6 acetaminophen          acute hepatic failure A: confirmed (literature + FAERS)       0.88       0.81         0.94     7.0      118

,rank,drug_generic,adr_pt,tier,composite,lit_score,faers_score,n_docs,n_strong,n_weak,...,n_event_mentions,mean_conf_positive,evidence,faers_a,PRR,ROR,EBGM,EB05,faers_signal,in_faers
0,1,metformin,lactic acidosis,A: confirmed (literature + FAERS),0.990,1.000,0.979,13.0,11.0,0.0,...,13.0,0.961,8.944,330,330.37,485.76,185.653,169.439,True,True
1,2,clopidogrel,procedural haemorrhage,A: confirmed (literature + FAERS),0.979,0.958,1.000,10.0,9.0,0.0,...,6.0,0.891,7.218,61,479.43,538.57,288.462,232.657,True,True
2,3,olanzapine,neuroleptic malignant syndrome,A: confirmed (literature + FAERS),0.948,0.979,0.917,10.0,9.0,0.0,...,10.0,0.947,7.668,28,119.29,124.63,98.018,71.167,True,True
3,4,omeprazole,hypomagnesaemia,A: confirmed (literature + FAERS),0.917,0.938,0.896,11.0,9.0,0.0,...,11.0,0.968,7.126,41,101.47,109.28,87.828,67.495,True,True
4,5,atorvastatin,immune-mediated myositis,A: confirmed (literature + FAERS),0.906,0.854,0.958,10.0,8.0,0.0,...,10.0,0.929,5.944,54,205.64,215.90,131.827,104.868,True,True
5,6,acetaminophen,acute hepatic failure,A: confirmed (literature + FAERS),0.875,0.812,0.938,7.0,6.0,0.0,...,7.0,0.937,4.817,118,193.54,214.68,125.917,107.974,True,True
6,7,atorvastatin,myopathy,A: confirmed (literature + FAERS),0.812,0.771,0.854,4.0,4.0,0.0,...,4.0,0.975,3.900,19,55.12,56.06,46.739,31.657,True,True
7,8,omeprazole,hypocalcaemia,C: FAERS-only / weak literature,0.792,0.750,0.833,6.0,6.0,0.0,...,0.0,0.792,2.375,28,49.61,52.15,45.580,33.109,True,True
8,9,acetaminophen,hepatic failure,A: confirmed (literature + FAERS),0.729,0.688,0.771,4.0,3.0,0.0,...,4.0,0.967,2.175,27,18.78,19.20,16.476,12.700,True,True
9,10,metformin,metabolic acidosis,C: FAERS-only / weak literature,0.719,0.562,0.875,5.0,5.0,0.0,...,0.0,0.790,1.975,102,66.45,73.68,57.258,48.523,True,True


## 6. Visualisation and Streamlit dashboard

The static figures are generated below. The interactive dashboard is implemented in `results/dashboard.py` and run from a terminal.

In [10]:
runpy.run_path(str(ROOT / "scripts" / "07_visualize.py"), run_name="__main__")

print("Streamlit dashboard: results/dashboard.py")
print("Run in a terminal: cd <project-folder> && streamlit run results/dashboard.py")
print("Open: http://localhost:8501")

wrote figures/fig1-3; run the Streamlit dashboard with: streamlit run results/dashboard.py
Streamlit dashboard: results/dashboard.py
Run in a terminal: cd <project-folder> && streamlit run results/dashboard.py
Open: http://localhost:8501
